# ChpWiseDB — Policy Compliance Pipeline

| Component | Model |
|-----------|-------|
| Embedding | BGE-large `BAAI/bge-large-en-v1.5` |
| Reranker  | Qwen3-Reranker-4B |
| LLM       | Mistral-7B-Instruct-v0.3 *(Section 9 — run after validation)* |

**`qwen_norm_score`**: yes/no log-softmax probability → absolute 0–1 relevance score

---

## Changes from previous version
1. Policy document chunking and vector DB removed
2. Parser extracts multiple standard codes and chapters
3. Three-tier routed retrieval: exact standard code lookup → chapter search → global
4. Answer generation compares top-3, top-5, top-7 chunks

---

## Run order
| Section | Purpose | Run now? |
|---------|---------|----------|
| 1 | Imports & Config | ✅ |
| 2 | Chunk & index JCI ESS | ✅ |
| 3 | Load BGE-large + retrieval | ✅ |
| 4 | Load Qwen reranker | ✅ |
| 5 | Query parser | ✅ |
| 6 | Gold standard queries | ✅ |
| 7 | Validation loop (no LLM) | ✅ |
| 8 | Compliance analysis (Mistral) | ⏸ after Section 7 |
| 9 | Summary | ✅ after Section 7 |

---
## Section 1 — Imports & Config
Policy PDF and DB name removed. All other settings unchanged.
Update `JCI_PDF` to match your local file path.

In [ ]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
import torch
import transformers

from collections import defaultdict, Counter
from dataclasses import dataclass, asdict

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Paths ────────────────────────────────────────────────────
JCI_PDF     = 'ESS Standards_7th Ed .pdf'
DB_SAVE_DIR = 'saved_dbs'

# ── Embedding model ──────────────────────────────────────────
EMBEDDING_MODEL_NAME   = 'BGE-large'
EMBEDDING_MODEL_STRING = 'BAAI/bge-large-en-v1.5'

# ── Reranker ─────────────────────────────────────────────────
QWEN_MODEL_ID = 'Qwen/Qwen3-Reranker-4B'

# ── LLM (Section 8 only) ──────────────────────────────────
LLM_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

# ── Retrieval settings ───────────────────────────────────────
PER_DB_K   = 8    # chunks fetched per chapter DB in vector search
FINAL_K    = 10   # chunks kept after global re-sort and dedup
MIN_CHUNKS = 7    # minimum chunks needed; Tier 1 supplements if below this

# ── LLM settings (Section 8 only) ───────────────────────────
LLM_MAX_TOKENS = 512

# ── JCI chapter codes ────────────────────────────────────────
VALID_CODES = {
    'APR', 'IPSG', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'MMU',
    'QPS', 'PCI',  'GLD', 'FMS', 'SQE', 'MOI', 'MPE', 'HRP',
}
END_OF_BOOK_MARKERS = ['glossary', 'index', 'appendix', 'references']

# ── Parser config ────────────────────────────────────────────
VALID_CHAPTERS = VALID_CODES


TASK_KEYWORDS = {
    'compliance_check': [
        'comply', 'complies', 'compliance', 'aligned', 'aligns',
        'meets', 'meet', 'satisfies', 'consistent with', 'in line with',
        'conform', 'gap', 'gaps', 'missing', 'is this', 'does this',
        'adequate', 'fulfil',
    ],
    'comparison': [
        'compare', 'comparison', 'versus', 'vs', 'differ',
        'difference', 'how does', 'contrast', 'against',
    ],
    'clarification': [
        'what does', 'what is', 'what are', 'explain', 'define',
        'describe', 'clarify', 'what requirement', 'what criteria',
        'what should', 'how should',
    ],
}

print('Section 1 complete — config loaded.')

---
## Section 2 — Load, Chunk & Build FAISS DB: JCI ESS
- Loads the JCI ESS PDF and groups pages by chapter
- Splits each chapter into standard-level chunks with `standard_code` metadata
- Builds one FAISS DB per chapter under `saved_dbs/BGE-large/<chapter>/`
- `chapter_chunks` dict is also kept in memory for Tier 1 direct lookup
- **Run once** — skips if DBs already exist

In [ ]:
def load_pdf(path: str):
    pages = PyPDFLoader(path).load()
    print(f'Loaded {len(pages)} pages from JCI PDF.')
    return pages


def detect_chapter(text: str):
    """Returns JCI chapter code if this page is a chapter title page."""
    for line in text.split('\n'):
        m = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line.strip())
        if m and m.group(1) in VALID_CODES:
            return m.group(1)
    return None


def is_end_of_book(text: str) -> bool:
    return any(marker in text.strip()[:200].lower() for marker in END_OF_BOOK_MARKERS)


def group_pages_by_chapter(pages) -> dict:
    chapter_pages = defaultdict(list)
    current = 'UNKNOWN'
    for page in pages:
        detected = detect_chapter(page.page_content)
        if detected:
            current = detected
        chapter_pages[current].append(page)

    if 'HRP' in chapter_pages:
        before = len(chapter_pages['HRP'])
        chapter_pages['HRP'] = [
            p for p in chapter_pages['HRP'] if not is_end_of_book(p.page_content)
        ]
        print(f'HRP: removed {before - len(chapter_pages["HRP"])} end-of-book pages.')

    print('\n=== Chapter page counts ===')
    for ch, pgs in chapter_pages.items():
        print(f'  {ch}: {len(pgs)} pages')
    return chapter_pages


def build_chapter_chunks(chapter_pages: dict, chunk_size: int = 1000,
                          chunk_overlap: int = 250) -> dict:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' '],
    )
    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        full_text = '\n'.join(p.page_content for p in pg_list)
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        toc = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
            full_text, re.IGNORECASE
        )
        if toc:
            full_text = full_text[toc.start():]

        pattern = (
            rf'(?:\n|^)(?:Standard|Requirement)\s+'
            rf'({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?(?:\s*\.\s*\d+)?)\b'
        )
        splits = re.split(pattern, full_text)

        docs = []
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={'chapter': chapter, 'standard_code': 'INTRO'},
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break
            code    = splits[i].strip().replace(' ', '')
            content = f'{code}\n{splits[i + 1]}'
            if len(content.strip()) < 100:
                continue
            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={'chapter': chapter, 'standard_code': code},
                ))
            else:
                for j, sub in enumerate(splitter.split_text(content)):
                    if len(sub.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=sub.strip(),
                        metadata={'chapter': chapter, 'standard_code': code, 'chunk_index': j},
                    ))

        chapter_chunks[chapter] = docs
        unique = len(set(d.metadata['standard_code'] for d in docs))
        print(f'  {chapter}: {len(docs)} chunks across {unique} standards')

    return chapter_chunks


pages          = load_pdf(JCI_PDF)
chapter_pages  = group_pages_by_chapter(pages)
chapter_chunks = build_chapter_chunks(chapter_pages)

# ── Build and save chapter-level FAISS DBs ───────────────────────
model_dir = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME)

if os.path.exists(model_dir) and any(
    os.path.isdir(os.path.join(model_dir, ch)) for ch in VALID_CODES
):
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs already exist — skipping build.')
else:
    print(f'[{EMBEDDING_MODEL_NAME}] Building JCI chapter DBs...')
    os.makedirs(model_dir, exist_ok=True)
    jci_build_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_STRING,
        encode_kwargs={'normalize_embeddings': True},
    )
    for chapter, docs in chapter_chunks.items():
        if docs:
            db = FAISS.from_documents(docs, jci_build_embeddings)
            db.save_local(os.path.join(model_dir, chapter))
            print(f'  Saved: {chapter}')
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs done.')

print('Section 2 complete.')

---
## Section 3 — Load BGE-large Embeddings & Retrieval Functions
- Loads BGE-large embedding model
- Loads all JCI chapter FAISS DBs into memory
- Defines `retrieve()` for vector search and `retrieve_by_standard_code()` for Tier 1 exact lookup
- Defines `route_and_retrieve()` with three-tier routing:
  - **Tier 1**: explicit standard code found → direct chunk lookup by metadata
  - **Tier 2**: chapter found → vector search within that chapter DB
  - **Tier 3**: nothing found → global vector search across all DBs

In [ ]:
print(f'Loading embeddings model: {EMBEDDING_MODEL_STRING}')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_STRING,
    encode_kwargs={'normalize_embeddings': True},
)

# ── Load JCI chapter DBs ─────────────────────────────────────
jci_dbs = {}
for chapter in chapter_chunks.keys():
    chapter_path = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME, chapter)
    if os.path.exists(chapter_path):
        jci_dbs[chapter] = FAISS.load_local(
            chapter_path, embeddings, allow_dangerous_deserialization=True,
        )
print(f'Loaded {len(jci_dbs)} JCI chapter DBs.')


def retrieve(query: str, dbs: dict) -> list:
    """
    Vector similarity search across given DBs.
    Returns list of (Document, l2_score) sorted by score ascending.
    Used by Tier 2 and Tier 3.
    """
    pool = []
    for chapter, db in dbs.items():
        for doc, score in db.similarity_search_with_score(query, k=PER_DB_K):
            doc.metadata['chapter'] = chapter
            pool.append((doc, score))

    pool.sort(key=lambda x: x[1])

    seen, unique = set(), []
    for doc, score in pool:
        key = (doc.page_content, doc.metadata.get('standard_code'))
        if key not in seen:
            unique.append((doc, score))
            seen.add(key)

    return unique[:FINAL_K]


def retrieve_by_standard_code(standard_codes: list, chapter_chunks: dict) -> list:
    """
    Tier 1: directly returns all chunks whose standard_code metadata
    matches any of the given codes. No vector search. Score = 0.0.
    
    e.g. ['ACC.5.1', 'AOP.1'] returns every chunk tagged with either code.
    """
    results = []
    for code in standard_codes:
        chapter = code.split('.')[0]   # 'ACC' from 'ACC.5.1'
        if chapter not in chapter_chunks:
            continue
        for doc in chapter_chunks[chapter]:
            if doc.metadata.get('standard_code') == code:
                results.append((doc, 0.0))  # 0.0 = exact match
    print(f'  Tier 1 direct lookup: {len(results)} chunks for {standard_codes}')
    return results


def route_and_retrieve(query: str, parsed, dbs: dict, chapter_chunks: dict, min_chunks: int = MIN_CHUNKS,) -> list:
    """
    Three-tier routing:

    Tier 1 — explicit standard code(s) found in query
             Directly looks up chunks by standard_code metadata.
             If fewer than min_chunks returned, supplements with
             chapter vector search to fill remaining slots.
             Exact-match chunks (score=0.0) always rank first.

    Tier 2 — chapter(s) found but no full standard code
             Vector search within each identified chapter DB.
             Multiple chapters are searched and merged.

    Tier 3 — nothing found
             Global vector search across all chapter DBs.
    """
    retrieval_query = query

    # ── Tier 1: exact standard code lookup ────────────────────────
    if parsed.all_standards:
        direct = retrieve_by_standard_code(parsed.all_standards, chapter_chunks)

        if len(direct) >= min_chunks:
            return direct

        # Not enough — supplement with chapter vector search
        print(f'  Tier 1 returned {len(direct)} chunks — supplementing with chapter search')
        seen = {(doc.page_content, doc.metadata.get('standard_code')) for doc, _ in direct}
        pool = list(direct)

        for chapter in parsed.all_chapters:
            if chapter in dbs:
                for doc, score in dbs[chapter].similarity_search_with_score(
                    retrieval_query, k=PER_DB_K
                ):
                    key = (doc.page_content, doc.metadata.get('standard_code'))
                    if key not in seen:
                        pool.append((doc, score))
                        seen.add(key)

        # Sort: exact matches (0.0) stay at top, similarity scores after
        pool.sort(key=lambda x: x[1])
        return pool[:FINAL_K]

    # ── Tier 2: chapter(s) known ─────────────────────────────────
    if parsed.all_chapters:
        pool = []
        seen = set()
        for chapter in parsed.all_chapters:
            if chapter in dbs:
                for doc, score in dbs[chapter].similarity_search_with_score(
                    retrieval_query, k=PER_DB_K
                ):
                    doc.metadata['chapter'] = chapter
                    key = (doc.page_content, doc.metadata.get('standard_code'))
                    if key not in seen:
                        pool.append((doc, score))
                        seen.add(key)
        if pool:
            pool.sort(key=lambda x: x[1])
            return pool[:FINAL_K]

    # ── Tier 3: global fallback ───────────────────────────────────
    print('  Tier 3 global search')
    return retrieve(retrieval_query, dbs)


print('Section 3 complete — embeddings and retrieval ready.')

---
## Section 4 — Load Qwen Reranker
- Loads `Qwen3-Reranker-4B` as a causal LM
- Scores each (query, chunk) pair as yes/no relevance
- `norm_score` = log-softmax yes-probability → absolute 0–1
- Runs on chunks from **all three tiers** — including Tier 1 exact matches
- For Tier 1: Qwen picks the most relevant sub-chunk when a standard has multiple chunks

In [ ]:
print(f'Loading Qwen reranker: {QWEN_MODEL_ID}')

qwen_r_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')
qwen_r_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID, device_map='auto'
).eval()

qwen_r_true_id  = qwen_r_tok.convert_tokens_to_ids('yes')
qwen_r_false_id = qwen_r_tok.convert_tokens_to_ids('no')
qwen_r_maxlen   = 8192

_qwen_prefix = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the '
    'Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
    '<|im_end|>\n<|im_start|>user\n'
)
_qwen_suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
qwen_r_prefix_toks = qwen_r_tok.encode(_qwen_prefix, add_special_tokens=False)
qwen_r_suffix_toks = qwen_r_tok.encode(_qwen_suffix, add_special_tokens=False)

_QWEN_INSTRUCTION = (
    'Given a hospital accreditation policy compliance query, retrieve the '
    'document chunk that contains the relevant JCI standard, intent, or '
    'measurable elements needed to assess compliance.'
)


def rerank_qwen(query: str, retrieved: list) -> list:
    """
    Returns list of (doc, norm_score) sorted by norm_score descending.
    norm_score = log_softmax yes-probability — absolute 0-1.
    Works identically whether chunks came from Tier 1, 2, or 3.
    """
    docs  = [doc for doc, _ in retrieved]
    pairs = [
        f'<Instruct>: {_QWEN_INSTRUCTION}\n'
        f'<Query>: {query}\n'
        f'<Document>: {doc.page_content[:400]}'
        for doc in docs
    ]

    inputs = qwen_r_tok(
        pairs, padding=False, truncation='longest_first',
        return_attention_mask=False,
        max_length=qwen_r_maxlen - len(qwen_r_prefix_toks) - len(qwen_r_suffix_toks),
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = qwen_r_prefix_toks + ele + qwen_r_suffix_toks
    inputs = qwen_r_tok.pad(
        inputs, padding=True, return_tensors='pt', max_length=qwen_r_maxlen
    )
    for key in inputs:
        inputs[key] = inputs[key].to(qwen_r_model.device)

    with torch.no_grad():
        logits = qwen_r_model(**inputs).logits[:, -1, :]
        tv     = logits[:, qwen_r_true_id]
        fv     = logits[:, qwen_r_false_id]
        scores = torch.nn.functional.log_softmax(
            torch.stack([fv, tv], dim=1), dim=1
        )[:, 1].exp().tolist()

    combined = [(doc, round(score, 4)) for doc, score in zip(docs, scores)]
    return sorted(combined, key=lambda x: x[1], reverse=True)


print('Section 4 complete — Qwen reranker loaded.')

---
## Section 5 — Query Parser
Updated to extract **multiple** standard codes and chapters from a single query.

| Field | What it extracts | Example |
|-------|------------------|---------|
| `task_type` | What the user wants | `compliance_check` |
| `target_policy` | Which hospital policy | `Transfer Policy` |
| `policy_clause` | Policy clause label | `5.1.4` |
| `reference_standard` | Primary JCI standard | `ACC.5.1` |
| `reference_chapter` | Primary JCI chapter | `ACC` |
| `all_standards` | **All** standards found | `['ACC.5.1', 'AOP.1']` |
| `all_chapters` | **All** chapters found | `['ACC', 'AOP']` |
| `measurable_element` | ME reference | `ME 3` |

**Routing priority:**
1. `all_standards` not empty → Tier 1 exact lookup
2. `all_chapters` not empty → Tier 2 chapter search
3. Nothing found → Tier 3 global

In [ ]:
@dataclass
class ParsedQuery:
    task_type:          str
    policy_clause:      str | None
    reference_standard: str | None   # primary (first found)
    reference_chapter:  str | None   # primary (first found)
    all_standards:      list          # all standard codes found
    all_chapters:       list          # all chapters found
    measurable_element: str | None
    retrieval_scope:    str           # 'targeted' / 'chapter' / 'global'
    routing_source:     str           # 'explicit' / 'inferred' / 'global'

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


def extract_jci_standard(text: str) -> tuple[list, list]:
    """
    Extracts ALL JCI standard codes and chapters found in the query.
    Returns (list of standard codes, list of unique chapters).

    e.g. 'ACC.5.1 and AOP.1' -> (['ACC.5.1', 'AOP.1'], ['ACC', 'AOP'])
    e.g. 'ACC requirements'   -> ([], ['ACC'])
    e.g. nothing found        -> ([], [])
    """
    standards = []
    chapters  = []

    # First pass: find full standard codes e.g. ACC.5.1
    pattern = r'\b([A-Z]{2,4})\.(\d+(?:\.\d+){0,2})\b'
    for chapter, sub in re.findall(pattern, text):
        if chapter in VALID_CHAPTERS:
            std_code = f'{chapter}.{sub}'
            if std_code not in standards:
                standards.append(std_code)
            if chapter not in chapters:
                chapters.append(chapter)

    # Second pass: find standalone chapter codes not already captured
    for chapter in VALID_CHAPTERS:
        if chapter not in chapters:
            if re.search(rf'\b{chapter}\b', text):
                chapters.append(chapter)

    return standards, chapters


def extract_policy_clause(text: str) -> str | None:
    """
    Extracts hospital policy clause numbers e.g. 5.1.4.
    Excludes numbers that are part of a JCI standard code (e.g. the 5.1 in ACC.5.1).
    """
    for match in re.finditer(r'\b(\d+\.\d+(?:\.\d+)?)\b', text):
        start = match.start()
        preceding = text[:start].rstrip()
        if re.search(r'[A-Z]{2,4}\.$', preceding):
            continue   # part of a JCI code like ACC.5.1 — skip
        return match.group(1)
    return None


def extract_measurable_element(text: str) -> str | None:
    match = re.search(r'\bmeasurable\s+elements?\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    match = re.search(r'\bme\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    return None


def detect_task_type(text: str) -> str:
    text_lower = text.lower()
    for task_type, keywords in TASK_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return task_type
    return 'general_qa'


def parse_query(query: str, print_json: bool = True) -> ParsedQuery:
    """
    Rule-based parser — no LLM call.
    Extracts all standard codes and chapters found in the query.
    Primary fields use the first found value for backward compatibility.
    all_standards and all_chapters drive the three-tier routing.
    """
    task_type          = detect_task_type(query)
    policy_clause      = extract_policy_clause(query)
    measurable_element = extract_measurable_element(query)
    standards, chapters = extract_jci_standard(query)

    primary_standard = standards[0] if standards else None
    primary_chapter  = chapters[0]  if chapters  else None

    if primary_standard:
        scope, source = 'targeted', 'explicit'
    elif primary_chapter:
        scope, source = 'chapter', 'inferred'
    else:
        scope, source = 'global', 'global'

    result = ParsedQuery(
        task_type          = task_type,
        policy_clause      = policy_clause,
        reference_standard = primary_standard,
        reference_chapter  = primary_chapter,
        all_standards      = standards,
        all_chapters       = chapters,
        measurable_element = measurable_element,
        retrieval_scope    = scope,
        routing_source     = source,
    )

    if print_json:
        print(result.to_json())

    return result


# ── Sanity check ───────────────────────────────────────────────
print('Parser sanity check:')
print('-' * 70)
_test_queries = [
    'Does clause 5.1 of our Transfer Policy comply with ACC.5.1?',
    'Compare our transfer process against ACC requirements',
    'What are the handover requirements for inter-hospital transfer?',
    'Does our policy meet Measurable Element 2 of ACC.5.1?',
    'How does Section 5.2.14 compare to JCI ACC.2 and AOP.2?',
]
for q in _test_queries:
    print(f'Query: {q}')
    parse_query(q, print_json=True)
    print()

print('Section 5 complete — parser ready.')

---
## Section 6 — Gold Standard Queries
Clinician-vetted policy compliance queries.
Replace placeholder entries with your actual Q&As before running Section 7.

Each entry needs: `query`, `policy_text`, `expected_standard`, `expected_chapter`, `task_type`

In [ ]:
policy_gold_standard = [
    {
        'query':             'The current Policy Purpose Statement (1.1) reads: "To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients." Does this purpose statement comply with JCI standards for continuity of care, and how does it compare to what JCI ACC.3 requires in terms of scope and integration across care settings?',
        'policy_text':       'To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients',
        'expected_standard': 'ACC.3',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 3.7 on inter-hospital transfer communication states: "communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team." Does this section meet the requirements of JCI ACC.5.1, and what gaps exist between this communication requirement and what the standard mandates in terms of content and documentation?',
        'policy_text':       'communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team.',
        'expected_standard': 'ACC.5.1',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.1.4 requires the referring team to "summarize the case history of patient and document it in SCM before transferring the patient to the other discipline," including health status, summary of care, patient progress, and reason for transfer. How does this documentation requirement compare to what JCI ACC.3 Measurable Element 5 specifies, and are there any elements missing that would constitute a compliance gap?',
        'policy_text':       ' ',
        'expected_standard': 'ACC.3',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.10 states that "critically ill patients requiring inter-hospital transfer should be accompanied by a doctor of at least MO or PGY2 Resident grade" who assumes responsibility for en-route medical control. Does this section fully satisfy JCI ACC.5 Measurable Element 3, and how does its scope of coverage compare to what the standard requires across different patient types?',
        'policy_text':       'critically ill patients requiring inter-hospital transfer should be accompanied by a doctor of at least MO or PGY2 Resident grade',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.11 states that "accompanying personnel shall be adequately equipped for any interventions required or anticipated," citing endotracheal intubation as an example. How does this approach to equipment planning compare to what JCI ACC.5 Measurable Element 4 requires, and does the current policy language meet the standards expectations for systematic documentation?',
        'policy_text':       'accompanying personnel shall be adequately equipped for any interventions required or anticipated',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.12 places responsibility on the accepting doctor to assess "that the patient requires transfer and is suitable for transfer before agreeing to transfer the patient." Does this section align with JCI ACC.5 intent regarding the basis for inter-hospital transfers, and what does the standard suggest should be defined at the policy level beyond the accepting doctors individual assessment?',
        'policy_text':       'that the patient requires transfer and is suitable for transfer before agreeing to transfer the patient.',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.13 states that "the accepting doctor is also responsible for coordinating with other disciplines for the patients total care." How does this compare to what JCI ACC.3 requires for multidisciplinary coordination, and does assigning responsibility to a single doctor satisfy the standards expectations for documented coordination processes?',
        'policy_text':       'the accepting doctor is also responsible for coordinating with other disciplines for the patients total care',
        'expected_standard': 'ACC.3',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.14 outlines the accepting doctors responsibilities for ensuring bed availability and reviewing the patient on arrival, including liaising with ICU or overflow ward doctors. How does this section compare to the requirements under JCI ACC.2 and AOP.2, and what gaps exist between the current policys approach to bed management and the standards expectation for a formalised, hospital-wide patient flow process?',
        'policy_text':       ' ',
        'expected_standard': 'ACC.2',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Section 5.2.15 states that "outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer." Does this exclusion align with JCI ACC.1 and AOP.1 requirements for patient assessment and care setting appropriateness, and what does the standard require in terms of formalising assessment processes across inpatient and outpatient settings?',
        'policy_text':       'outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer.',
        'expected_standard': 'ACC.1',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
]

print(f'Policy gold standard: {len(policy_gold_standard)} queries loaded.')
print('Section 6 complete.')

---
## Section 7 — Validation Loop (Retrieval Only — No LLM)
For each query:
1. Parse → routing decision (with `all_standards` and `all_chapters`)
2. `route_and_retrieve()` → three-tier routing
3. Qwen reranker → scores chunks
4. Record top-1 and top-3 matches vs expected

Results saved to `PARSER_VALIDATION.csv`.

In [ ]:
# ── Shared fields for all 4 files ─────────────────────────────
# These are the same regardless of how many chunks are passed
rows_top1 = []
rows_top3 = []
rows_top5 = []
rows_top7 = []

for i, item in enumerate(policy_gold_standard, 1):
    query        = item['query']
    expected_std = item['expected_standard']
    expected_ch  = item['expected_chapter']

    print(f'[{i:02d}/{len(policy_gold_standard)}] Expected: {expected_std}')

    # 1. Parse
    parsed = parse_query(query, print_json=False)

    # 2. Three-tier route + retrieve
    retrieved = route_and_retrieve(query, parsed, jci_dbs, chapter_chunks)

    # 3. Rerank with Qwen
    qwen_ranked = rerank_qwen(query, retrieved)

    # ── Slice each depth ──────────────────────────────────────
    top1_doc,  top1_score  = qwen_ranked[0]
    top3       = qwen_ranked[:3]
    top5       = qwen_ranked[:5]
    top7       = qwen_ranked[:7]

    top1_std   = top1_doc.metadata.get('standard_code')
    top3_stds  = [doc.metadata.get('standard_code') for doc, _ in top3]
    top5_stds  = [doc.metadata.get('standard_code') for doc, _ in top5]
    top7_stds  = [doc.metadata.get('standard_code') for doc, _ in top7]

    top3_scores = [round(s, 4) for _, s in top3]
    top5_scores = [round(s, 4) for _, s in top5]
    top7_scores = [round(s, 4) for _, s in top7]

    # ── Accuracy flags ────────────────────────────────────────
    top1_match = top1_std == expected_std
    top3_match = expected_std in str(top3_stds)
    top5_match = expected_std in str(top5_stds)
    top7_match = expected_std in str(top7_stds)

    # ── Shared metadata fields ────────────────────────────────
    shared = {
        'query':                     query,
        'expected_standard':         expected_std,
        'expected_chapter':          expected_ch,
        'gold_task_type':            item['task_type'],
        'parsed_task_type':          parsed.task_type,
        'parsed_chapter':            parsed.reference_chapter,
        'parsed_all_standards':      str(parsed.all_standards),
        'parsed_all_chapters':       str(parsed.all_chapters),
        'parsed_measurable_element': parsed.measurable_element,
        'routing_source':            parsed.routing_source,
        'retrieval_scope':           parsed.retrieval_scope,
    }

    # ── Top-1 row ─────────────────────────────────────────────
    rows_top1.append({
        **shared,
        'retrieved_standard':  top1_std,
        'retrieved_score':     top1_score,
        'retrieved_chunks':    top1_doc.page_content.strip(),
        'top1_match':          top1_match,
    })

    # ── Top-3 row ─────────────────────────────────────────────
    rows_top3.append({
        **shared,
        'retrieved_standards': str(top3_stds),
        'retrieved_scores':    str(top3_scores),
        'retrieved_chunks':    '\n\n---\n\n'.join(
                                   doc.page_content.strip() for doc, _ in top3
                               ),
        'top1_match':          top1_match,
        'top3_match':          top3_match,
    })

    # ── Top-5 row ─────────────────────────────────────────────
    rows_top5.append({
        **shared,
        'retrieved_standards': str(top5_stds),
        'retrieved_scores':    str(top5_scores),
        'retrieved_chunks':    '\n\n---\n\n'.join(
                                   doc.page_content.strip() for doc, _ in top5
                               ),
        'top1_match':          top1_match,
        'top5_match':          top5_match,
    })

    # ── Top-7 row ─────────────────────────────────────────────
    rows_top7.append({
        **shared,
        'retrieved_standards': str(top7_stds),
        'retrieved_scores':    str(top7_scores),
        'retrieved_chunks':    '\n\n---\n\n'.join(
                                   doc.page_content.strip() for doc, _ in top7
                               ),
        'top1_match':          top1_match,
        'top7_match':          top7_match,
    })

    # ── Print progress ────────────────────────────────────────
    status = '\u2713' if top1_match else '\u2717'
    print(
        f'  [{status}] Got: {top1_std} '
        f'(score: {top1_score:.4f}) | '
        f'tier: {parsed.routing_source} | '
        f'all_standards: {parsed.all_standards}'
    )

# ── Save all 4 files ──────────────────────────────────────────
pd.DataFrame(rows_top1).to_csv('VALIDATION_TOP1.csv', index=False)
pd.DataFrame(rows_top3).to_csv('VALIDATION_TOP3.csv', index=False)
pd.DataFrame(rows_top5).to_csv('VALIDATION_TOP5.csv', index=False)
pd.DataFrame(rows_top7).to_csv('VALIDATION_TOP7.csv', index=False)

print(f'\nSaved 4 validation files ({len(rows_top1)} rows each):')
print('  VALIDATION_TOP1.csv')
print('  VALIDATION_TOP3.csv')
print('  VALIDATION_TOP5.csv')
print('  VALIDATION_TOP7.csv')

# ── Summary using top1 as reference ──────────────────────────
val_df = pd.DataFrame(rows_top1)   # used by Section 9 summary

print(f'\nQuick accuracy summary:')
print(f'  Top-1 : {pd.DataFrame(rows_top1)["top1_match"].mean()*100:.1f}%')
print(f'  Top-3 : {pd.DataFrame(rows_top3)["top3_match"].mean()*100:.1f}%')
print(f'  Top-5 : {pd.DataFrame(rows_top5)["top5_match"].mean()*100:.1f}%')
print(f'  Top-7 : {pd.DataFrame(rows_top7)["top7_match"].mean()*100:.1f}%')

print('Section 7 complete.')

---
## Section 8 — Compliance Analysis (Mistral)
> ⏸ Only run after Section 7 retrieval accuracy is satisfactory.

Generates three answers per query using top-3, top-5, and top-7 chunks.
This lets you compare which context window gives the best compliance verdict.

Results saved to `COMPLIANCE_ANALYSIS.csv`.

In [ ]:
COMPLIANCE_SYSTEM_PROMPT = """
You are an expert healthcare accreditation assistant specialised in the
Joint Commission International (JCI) 7th Edition Standards (special edition
for Singapore Ministry of Health).

You will be given:
- A user query
- One or more retrieved JCI standard chunks (the REFERENCE)
- 

YOUR PRIMARY RULE:
Assess compliance using ONLY the information explicitly stated in the
provided JCI chunk text. Do not use external knowledge, assumptions,
or prior familiarity with JCI standards.

OUTPUT FORMAT:
1. COMPLIANCE STATUS: Compliant / Partially Compliant / Non-Compliant / Insufficient Information
2. JUSTIFICATION: What in the policy text aligns or conflicts with the JCI standard
3. GAPS (if any): Specific requirements in the JCI standard not addressed by the policy
4. JCI REFERENCE: The standard code(s) from the retrieved chunks

PROHIBITED BEHAVIORS:
- Do NOT hallucinate standards or requirements not in the retrieved text
- Do NOT cite clauses or editions unless they appear in the chunk text
- Do NOT provide clinical or legal advice beyond what is stated

If the retrieved chunks do not contain sufficient information to assess
compliance, state: \"The provided JCI ESS excerpts do not contain sufficient
information to assess compliance with this clause.\"
"""


def build_compliance_prompt(query: str, chunks, task_type: str = None) -> list:
    if isinstance(chunks, Document):
        chunks = [chunks]
    context_parts = []
    for idx, chunk in enumerate(chunks, 1):
        context_parts.append(
            f'[JCI Chunk {idx} | Standard: {chunk.metadata.get("standard_code", "?")} | '
            f'Chapter: {chunk.metadata.get("chapter", "?")}]\n'
            f'{chunk.page_content.strip()}'
        )
    context_str = '\n\n'.join(context_parts)
    user_content = (
        f'TASK: {parsed.task_type}\n\n' 
        f'RELEVANT JCI STANDARDS:\n{context_str}\n\n'
        f'COMPLIANCE QUESTION:\n{query}\n\n'
        f'Assessment (cite the JCI standard code where relevant):'
    )
    return [
        {'role': 'system', 'content': COMPLIANCE_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_content},
    ]


def generate_mistral_answer(messages: list) -> str:
    output = mistral_pipeline(messages, max_new_tokens=LLM_MAX_TOKENS)
    return output[0]['generated_text'][-1]['content'].strip()


print(f'Loading LLM: {LLM_MODEL_ID}')
mistral_pipeline = transformers.pipeline(
    'text-generation', model=LLM_MODEL_ID,
    model_kwargs={'torch_dtype': torch.bfloat16},
    device_map='auto',
)
print(f'LLM loaded: {LLM_MODEL_ID}')

In [ ]:
# ── Compliance generation loop ──────────────────────────────
compliance_rows = []

for i, item in enumerate(policy_gold_standard, 1):
    query        = item['query']
    expected_std = item['expected_standard']

    print(f'[{i:02d}/{len(policy_gold_standard)}] Generating: {expected_std}')

    parsed      = parse_query(query, print_json=False)
    retrieved   = route_and_retrieve(query, parsed, jci_dbs, chapter_chunks)
    rerank_q    = query
    qwen_ranked = rerank_qwen(rerank_q, retrieved)

    # ── Slice top 3, 5, 7 ────────────────────────────────────
    top3_docs   = [doc for doc, _ in qwen_ranked[:3]]
    top5_docs   = [doc for doc, _ in qwen_ranked[:5]]
    top7_docs   = [doc for doc, _ in qwen_ranked[:7]]

    top3_stds   = [doc.metadata.get('standard_code') for doc in top3_docs]
    top5_stds   = [doc.metadata.get('standard_code') for doc in top5_docs]
    top7_stds   = [doc.metadata.get('standard_code') for doc in top7_docs]

    top3_scores = [round(s, 4) for _, s in qwen_ranked[:3]]
    top5_scores = [round(s, 4) for _, s in qwen_ranked[:5]]
    top7_scores = [round(s, 4) for _, s in qwen_ranked[:7]]

    print('  Generating top-3 answer...')
    answer_top3 = generate_mistral_answer(build_compliance_prompt(query, top3_docs))
    print('  Generating top-5 answer...')
    answer_top5 = generate_mistral_answer(build_compliance_prompt(query, top5_docs))
    print('  Generating top-7 answer...')
    answer_top7 = generate_mistral_answer(build_compliance_prompt(query, top7_docs))

    compliance_rows.append({
        'query':            query,
        'expected_standard': expected_std,
        'routing_source':   parsed.routing_source,
        'all_standards':    str(parsed.all_standards),
        # ── Top-3 ──────────────────────────────────────────
        'top3_standards':   str(top3_stds),
        'top3_scores':      str(top3_scores),
        'top3_chunks':      '\n\n---\n\n'.join(d.page_content.strip() for d in top3_docs),
        'answer_top3':      answer_top3,
        # ── Top-5 ──────────────────────────────────────────
        'top5_standards':   str(top5_stds),
        'top5_scores':      str(top5_scores),
        'top5_chunks':      '\n\n---\n\n'.join(d.page_content.strip() for d in top5_docs),
        'answer_top5':      answer_top5,
        # ── Top-7 ──────────────────────────────────────────
        'top7_standards':   str(top7_stds),
        'top7_scores':      str(top7_scores),
        'top7_chunks':      '\n\n---\n\n'.join(d.page_content.strip() for d in top7_docs),
        'answer_top7':      answer_top7,
    })

compliance_df = pd.DataFrame(compliance_rows)
compliance_df.to_csv('COMPLIANCE_ANALYSIS.csv', index=False)
print(f'Saved: COMPLIANCE_ANALYSIS.csv ({len(compliance_df)} rows)')
print('Section 8 complete.')

---
## Section 9 — Summary
Run after Section 7. Compliance analysis results are in `COMPLIANCE_ANALYSIS.csv`.

In [ ]:
print('\n' + '='*60)
print('PARSER VALIDATION SUMMARY')
print('='*60)
print(f'Total queries: {len(val_df)}')
print(f'Top-1 accuracy: {val_df["top1_match"].mean()*100:.1f}%')

print('\nAccuracy by routing source:')
print(val_df.groupby('routing_source')[['top1_match']].mean().round(3).to_string())

print('\nAccuracy by retrieval scope:')
print(val_df.groupby('retrieval_scope')[['top1_match']].mean().round(3).to_string())

print('\nAccuracy by task type:')
print(val_df.groupby('parsed_task_type')[['top1_match']].mean().round(3).to_string())

print('\nFailed cases:')
failed = val_df[~val_df['top1_match']]
if len(failed) == 0:
    print('  None — all top-1 correct!')
else:
    for _, row in failed.iterrows():
        print(
            f"  Expected : {row['expected_standard']}"
            f" | Got    : {row['retrieved_standard']}"
            f" | Score  : {row['retrieved_score']}"
            f" | Tier   : {row['routing_source']}"
            f" | Standards found: {row['parsed_all_standards']}"
        )

print('\nQwen Score Summary:')
print(f"  Top-1 avg score: {val_df['retrieved_score'].mean():.4f}")
print()
print('If Qwen gives high scores but low accuracy it is confidently wrong.')
print('='*60)